In [1]:
import polars as pl 

In [2]:
data = pl.read_csv(r"C:\Users\Miguel\Documents\Proyecto_Grafico\Data\reporte_detallado_jugadores_final.csv")

In [3]:
df = (
    data
    .with_columns(
        pl.col("date_evento").str.to_datetime(strict=False).alias("date_evento")
    )
    .with_columns(
        pl.col("date_evento").dt.truncate("1mo").alias("mes")
    )
)

# Columnas por tipo
num_cols = [c for c, t in df.schema.items() if t in (pl.Int8, pl.Int16, pl.Int32, pl.Int64, pl.UInt8, pl.UInt16, pl.UInt32, pl.UInt64, pl.Float32, pl.Float64)]
str_cols = [c for c, t in df.schema.items() if t == pl.Utf8]

# Excluir columnas que no deben agregarse así
for c in ["date_evento", "mes", "agente_username"]:
    if c in num_cols: num_cols.remove(c)
    if c in str_cols: str_cols.remove(c)

aggs = [
    pl.len().alias("filas"),
    pl.col("date_evento").min().alias("fecha_min"),
    pl.col("date_evento").max().alias("fecha_max"),
]

# Nulos por columna (excluyendo mes y agente_username para no duplicar “nulos” obvios)
aggs += [pl.col(c).null_count().alias(f"{c}__nulos") for c in df.columns if c not in ("mes", "agente_username")]

# Métricas numéricas
aggs += [pl.col(c).sum().alias(f"{c}__suma") for c in num_cols]
aggs += [pl.col(c).mean().alias(f"{c}__prom") for c in num_cols]
aggs += [pl.col(c).min().alias(f"{c}__min") for c in num_cols]
aggs += [pl.col(c).max().alias(f"{c}__max") for c in num_cols]

# Cardinalidad para strings
aggs += [pl.col(c).n_unique().alias(f"{c}__n_unicos") for c in str_cols]

reporte_mensual_agencia = (
    df.group_by(["mes", "agente_username"])
      .agg(aggs)
      .sort(["mes", "agente_username"])
      .with_columns(pl.col("mes").dt.strftime("%Y-%m").alias("mes"))
)

reporte_mensual_agencia

mes,agente_username,filas,fecha_min,fecha_max,agente_id__nulos,player_id__nulos,player_internal_id__nulos,player_username__nulos,amount_bet_deportiva__nulos,amount_bet_casino__nulos,ggr_deportiva__nulos,ggr_casino__nulos,ggr_total__nulos,deportiva_tickets__nulos,casino_tickets__nulos,n_deposito__nulos,deposito__nulos,n_retiro__nulos,retiro__nulos,date_evento__nulos,porcentaje_propio__nulos,Tipo_Agente__nulos,comis_calculada__nulos,ngr_total__nulos,agente_id__suma,player_id__suma,player_internal_id__suma,amount_bet_deportiva__suma,amount_bet_casino__suma,ggr_deportiva__suma,ggr_casino__suma,ggr_total__suma,deportiva_tickets__suma,casino_tickets__suma,n_deposito__suma,deposito__suma,…,ngr_total__prom,agente_id__min,player_id__min,player_internal_id__min,amount_bet_deportiva__min,amount_bet_casino__min,ggr_deportiva__min,ggr_casino__min,ggr_total__min,deportiva_tickets__min,casino_tickets__min,n_deposito__min,deposito__min,n_retiro__min,retiro__min,porcentaje_propio__min,comis_calculada__min,ngr_total__min,agente_id__max,player_id__max,player_internal_id__max,amount_bet_deportiva__max,amount_bet_casino__max,ggr_deportiva__max,ggr_casino__max,ggr_total__max,deportiva_tickets__max,casino_tickets__max,n_deposito__max,deposito__max,n_retiro__max,retiro__max,porcentaje_propio__max,comis_calculada__max,ngr_total__max,player_username__n_unicos,Tipo_Agente__n_unicos
str,str,u32,datetime[μs],datetime[μs],u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,f64,f64,i64,f64,f64,f64,f64,f64,f64,f64,f64,f64,…,f64,f64,f64,i64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,i64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,u32,u32
"""2025-07""","""CCAA""",23,2025-07-07 00:00:00,2025-07-30 00:00:00,0,3,0,3,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,16238.0,14520.0,913813,734.32,0.0,-616.91,0.0,-616.91,238.0,0.0,19.0,188.53,…,-26.822174,706.0,726.0,39731,0.0,0.0,-303.43,0.0,-303.43,0.0,0.0,0.0,0.0,0.0,-236.0,20.0,-60.686,-303.43,706.0,726.0,39731,305.75,0.0,8.0,0.0,8.0,56.0,0.0,3.0,50.5,11.0,0.0,20.0,1.6,8.0,2,1
"""2025-07""","""CETBET""",11,2025-07-07 00:00:00,2025-07-31 00:00:00,0,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,10956.0,1620.0,435468,171.0,0.0,166.0,0.0,166.0,16.0,0.0,15.0,266.0,…,15.090909,996.0,162.0,39588,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,5.0,0.0,-100.0,20.0,0.0,0.0,996.0,162.0,39588,50.0,0.0,50.0,0.0,50.0,2.0,0.0,2.0,100.0,1.0,0.0,20.0,10.0,50.0,2,1
"""2025-07""","""CelsoF35""",1,2025-07-28 00:00:00,2025-07-28 00:00:00,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,3642.0,1590.0,6462,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,…,0.0,3642.0,1590.0,6462,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.0,0.003,0.0,3642.0,1590.0,6462,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.0,0.003,0.0,1,1
"""2025-07""","""ELRINCONDETEDDY""",1,2025-07-17 00:00:00,2025-07-17 00:00:00,0,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,6824.0,0.0,45410,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,…,0.0,6824.0,null,45410,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,5.0,0.0,0.0,6824.0,null,45410,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,5.0,0.0,0.0,1,1
"""2025-07""","""Ecuador21""",13,2025-07-04 00:00:00,2025-07-30 00:00:00,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,40053.0,64415.0,28080,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,…,0.0,3081.0,4955.0,2160,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,5.0,-0.3815,0.0,3081.0,4955.0,2160,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,5.0,0.5,0.0,1,1
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""2026-02""","""espinbet""",5,2026-02-01 00:00:00,2026-02-04 00:00:00,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,6255.0,51257.0,64463,36.49,0.0,35.48,0.0,35.48,25.0,0.0,4.0,25.0,…,7.096,1251.0,6769.0,12877,5.0,0.0,3.99,0.0,3.99,1.0,0.0,0.0,0.0,0.0,0.0,35.0,1.3965,3.99,1251.0,12573.0,12903,11.41,0.0,11.41,0.0,11.41,10.0,0.0,1.0,9.0,0.0,0.0,35.0,3.9935,11.41,2,1
"""2026-02""","""flippermex""",1,2026-02-01 00:00:00,2026-02-01 00:00:0